# Start here: improve a duration parser

Our time tracker thinks four minutes means four seconds. In five short cells,
you'll measure the bug, try a correction, and use the selected parser.
The correction is handwritten; Python runs the checks. No model or API key is needed.

**Start with a version → propose a change → test it → inspect what happened.**



Run the five cells in order. Every definition is below.
After installation, the example runs offline.

<a id="start-here-build-an-improvement-loop"></a>
<a id="improve-a-duration-parser"></a>
<a id="try-it-improve-a-prompt"></a>

## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

<a id="2-start-with-something-incomplete"></a>

## 2. Try the starting parser

`SEED` holds the parser's Python source. `load_parser` executes that source and
returns its function. Try `"4m"`: this version reads the number and ignores the unit.

In [ ]:
import meta_evolve as meta

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]


print("4m:", load_parser(SEED)("4m"), "seconds")
# Output:
# 4m: 4 seconds

<a id="define-what-correct-means"></a>
<a id="run-the-seed-check"></a>
<a id="inspect-the-grader"></a>
<a id="3-define-how-to-measure-it"></a>

## 3. Measure it with six checks

`evaluate` returns the fraction of correct answers; higher is better.
Only the two seconds cases pass. These expected answers stay fixed while we
try a revised parser.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def evaluate(source):
    parse = load_parser(source)
    passed = 0
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is int and actual == expected:
            passed += 1
    return passed / len(CASES)


print(f"Starting: {evaluate(SEED) * len(CASES):.0f}/{len(CASES)} checks")
# Output:
# Starting: 2/6 checks

<a id="4-simulate-an-agent"></a>

## 4. Propose a correction

The missing step is converting minutes and hours to seconds. `propose` receives
the current source and returns a proposed revision. Here it always returns this
handwritten correction; you can replace it with a model call later.

In [ ]:
def propose(source):
    return '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

<a id="four-inputs-one-result"></a>
<a id="assemble-the-improvement-loop"></a>
<a id="improvement-ties-and-regressions"></a>
<a id="5-run-two-revisions"></a>
<a id="6-see-what-changed"></a>
<a id="understand-the-result"></a>

## 5. Improve, inspect, and use it

`meta.improve` evaluates the starting parser, tries one revision (`trials=1`),
and keeps the version with the better score. `result.best().value` gives you
its source, ready to load and call. You set the checks and the attempt limit;
the run stops after that one revision attempt.

In [ ]:
result = meta.improve(
    seed=SEED,
    # This example returns a handwritten correction.
    # Replace it with a function that asks an agent or LLM for new candidates.
    proposer=propose,
    evaluator=evaluate,
    trials=1,
)

summary = result.summary()
print(f"Starting: {evaluate(SEED) * len(CASES):.0f}/{len(CASES)} checks")
print(f"Selected: {summary.primary_score * len(CASES):.0f}/{len(CASES)} checks")
print("Selected source:")
print(result.best().value, end="")

parse_seconds = load_parser(result.best().value)
print("4m:", parse_seconds("4m"), "seconds")
# Output:
# Starting: 2/6 checks
# Selected: 6/6 checks
# Selected source:
# def parse_seconds(text):
#     quantity = int(text[:-1])
#     seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
#     return quantity * seconds_per_unit[text[-1]]
# 4m: 240 seconds

Four minutes now gives **240 seconds**, and the selected parser passes **6/6**
checks. A tie or a worse revision keeps the earlier winner. These checks cover
whole-number durations ending in `s`, `m`, or `h`; passing them doesn't establish
correctness for every possible input.

`result` keeps the attempted versions and their outcomes in memory. To keep
that history after Python closes, [save and reopen the run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/).
Selecting a parser does not deploy it.

## Change and predict

Change `trials=1` to `trials=0` in cell 5 and rerun that cell. Predict the answer
before running it: **2/6** checks, and `"4m"` gives **4 seconds** again. With no
proposal attempts, the starting parser stays selected. Restore `trials=1`.

To try another check, add `("4m", 240)` to `CASES` in cell 3 and rerun cells 3
and 5. The starting score becomes **2/7** and the selected score **7/7**.
Rerunning all five cells repeats the experiment from the starting parser.

<a id="what-can-you-build"></a>
<a id="complete-source"></a>
<a id="keep-the-work"></a>
<a id="report-your-own-cost"></a>
<a id="run-with-a-model"></a>
<a id="the-same-loop-a-different-artifact"></a>
<a id="verification-and-remaining-work"></a>

## Next: draw, critique, and revise an SVG

[**Build a lighthouse with Feedback Descent**](https://sentient-xyz.github.io/meta-evolve-docs/build-patterns/feedback-descent/).
Keep the same loop, and replace the handwritten revision with a live agent.
A separate agent grades the rendered image and supplies feedback for the next
revision. The walkthrough includes a complete notebook and needs a Codex login.

- Stay with Python: [let a model improve this parser](https://sentient-xyz.github.io/meta-evolve-docs/guides/live-parser/).
- Adapt the experiment: [change the checks](https://sentient-xyz.github.io/meta-evolve-docs/guides/evaluation/) or choose a
  [task guide](https://sentient-xyz.github.io/meta-evolve-docs/guides/).
- Find another application: [explore all examples](https://sentient-xyz.github.io/meta-evolve-docs/examples/).

For a saved documentation build, use its matching
[package ZIP](https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip) with the local installation route above.